In [ ]:
# Cell 1: Cài đặt thư viện (chạy 1 lần)
print("="*80)
print("CÀI ĐẶT THƯ VIỆN CHO BÁO CÁO")
print("="*80)

import subprocess
import sys

# Danh sách các thư viện cần thiết
required_packages = ['pandas', 'numpy', 'openpyxl', 'matplotlib', 'seaborn', 'reportlab']

def check_and_install(package):
    """Kiểm tra và cài đặt package nếu chưa có"""
    try:
        __import__(package.replace('-', '_'))
        print(f"✅ {package} đã được cài đặt")
        return True
    except ImportError:
        print(f"📦 Đang cài đặt {package}...")
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", package, "--quiet"])
            print(f"✅ Đã cài đặt {package}")
            return True
        except Exception as e:
            print(f"❌ Lỗi cài đặt {package}: {e}")
            return False

print("🔍 Đang kiểm tra thư viện...")
for package in required_packages:
    check_and_install(package)

print("\n✅ Hoàn tất kiểm tra và cài đặt thư viện!")

# Cell 2: Import thư viện
print("\n" + "="*80)
print("XUẤT BÁO CÁO TỰ ĐỘNG")
print("Tạo báo cáo tổng hợp dạng Excel, PDF và Markdown")
print("="*80)

import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
import warnings
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils.dataframe import dataframe_to_rows
import matplotlib.pyplot as plt
import seaborn as sns

# Import reportlab sau khi đã cài đặt
try:
    from reportlab.lib import colors
    from reportlab.lib.pagesizes import A4, landscape
    from reportlab.platypus import SimpleDocTemplate, Table, TableStyle, Paragraph, Spacer, Image
    from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
    from reportlab.lib.units import inch
    REPORTLAB_AVAILABLE = True
except ImportError:
    REPORTLAB_AVAILABLE = False
    print("⚠️ ReportLab không khả dụng, sẽ bỏ qua xuất PDF")

import io
import os

warnings.filterwarnings('ignore')

print("✅ Đã import thư viện thành công!")

# Cell 3: Đường dẫn
print("\n" + "="*80)
print("XÁC ĐỊNH ĐƯỜNG DẪN")
print("="*80)

PROJECT_DIR = Path.cwd() / 'vietnam_retail_analysis'
DATA_DIR = PROJECT_DIR / 'data' / 'processed'
OUTPUTS_DIR = PROJECT_DIR / 'outputs'
REPORTS_DIR = PROJECT_DIR / 'reports'
MODELS_DIR = OUTPUTS_DIR / 'models'

# Tạo thư mục reports nếu chưa có
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"📁 Thư mục dự án: {PROJECT_DIR}")
print(f"📁 Thư mục dữ liệu: {DATA_DIR}")
print(f"📁 Thư mục kết quả: {OUTPUTS_DIR}")
print(f"📁 Thư mục models: {MODELS_DIR}")
print(f"📁 Thư mục báo cáo: {REPORTS_DIR}")

# Cell 4: Đọc dữ liệu
print("\n" + "="*80)
print("ĐỌC DỮ LIỆU")
print("="*80)

def load_all_data():
    """Đọc tất cả dữ liệu cần thiết"""
    data = {}
    
    # 1. Dữ liệu bán hàng
    sales_file = DATA_DIR / 'combined_sales_data.csv'
    if sales_file.exists():
        data['sales'] = pd.read_csv(sales_file, encoding='utf-8-sig')
        print("✅ Đã đọc dữ liệu bán hàng")
    else:
        print("⚠️ Không tìm thấy dữ liệu bán hàng")
        data['sales'] = None
    
    # 2. Dữ liệu phân cụm
    segments_file = OUTPUTS_DIR / 'customer_segments.csv'
    if segments_file.exists():
        data['segments'] = pd.read_csv(segments_file, encoding='utf-8-sig')
        print("✅ Đã đọc dữ liệu phân cụm")
    else:
        print("⚠️ Không tìm thấy dữ liệu phân cụm")
        data['segments'] = None
    
    # 3. Dữ liệu luật kết hợp
    rules_file = OUTPUTS_DIR / 'vietnam_retail_rules.csv'
    if rules_file.exists():
        data['rules'] = pd.read_csv(rules_file, encoding='utf-8-sig')
        print("✅ Đã đọc dữ liệu luật kết hợp")
    else:
        print("⚠️ Không tìm thấy dữ liệu luật kết hợp")
        data['rules'] = None
    
    # 4. Dữ liệu doanh số tháng
    monthly_file = OUTPUTS_DIR / 'monthly_sales.csv'
    if monthly_file.exists():
        data['monthly'] = pd.read_csv(monthly_file, encoding='utf-8-sig')
        print("✅ Đã đọc dữ liệu doanh số tháng")
    else:
        print("⚠️ Không tìm thấy dữ liệu doanh số tháng")
        data['monthly'] = None
    
    # 5. Dữ liệu so sánh mô hình
    model_comp_file = OUTPUTS_DIR / 'model_comparison.csv'
    if model_comp_file.exists():
        data['model_comp'] = pd.read_csv(model_comp_file, encoding='utf-8-sig')
        print("✅ Đã đọc dữ liệu so sánh mô hình")
    else:
        print("⚠️ Không tìm thấy dữ liệu so sánh mô hình")
        data['model_comp'] = None
    
    # 6. Dữ liệu bán giám sát
    semi_supervised_files = list(MODELS_DIR.glob('semi_supervised_*.pkl'))
    data['semi_supervised'] = len(semi_supervised_files)
    print(f"✅ Tìm thấy {len(semi_supervised_files)} mô hình bán giám sát")
    
    return data

data = load_all_data()

# Cell 5: Tạo báo cáo Excel
print("\n" + "="*80)
print("TẠO BÁO CÁO EXCEL")
print("="*80)

def create_excel_report(data, output_path):
    """Tạo báo cáo Excel với nhiều sheets"""
    
    wb = openpyxl.Workbook()
    
    # Định dạng
    header_font = Font(bold=True, color="FFFFFF")
    header_fill = PatternFill(start_color="366092", end_color="366092", fill_type="solid")
    center_alignment = Alignment(horizontal="center", vertical="center")
    border = Border(
        left=Side(style='thin'),
        right=Side(style='thin'),
        top=Side(style='thin'),
        bottom=Side(style='thin')
    )
    
    # Sheet 1: Tổng quan
    ws1 = wb.active
    ws1.title = "Tổng quan"
    
    overview_data = [
        ["CHỈ SỐ", "GIÁ TRỊ"],
        ["Ngày báo cáo", datetime.now().strftime("%d/%m/%Y %H:%M")],
    ]
    
    if data['sales'] is not None:
        overview_data.extend([
            ["Tổng số giao dịch", f"{len(data['sales']):,}"],
            ["Tổng doanh số", f"{data['sales']['Doanh số'].sum():,.0f} VNĐ"],
            ["Số khách hàng", f"{data['sales']['Mã khách hàng'].nunique():,}"],
            ["Số sản phẩm", f"{data['sales']['Mã sản phẩm'].nunique() if 'Mã sản phẩm' in data['sales'].columns else 0:,}"],
        ])
    
    if data['segments'] is not None:
        overview_data.append(["Số cụm khách hàng", f"{data['segments']['Cluster'].nunique()}"])
    
    if data['rules'] is not None:
        overview_data.append(["Số luật kết hợp", f"{len(data['rules'])}"])
    
    overview_data.append(["Số mô hình bán giám sát", f"{data['semi_supervised']}"])
    
    for row in overview_data:
        ws1.append(row)
    
    # Định dạng header
    for cell in ws1[1]:
        cell.font = header_font
        cell.fill = header_fill
        cell.alignment = center_alignment
        cell.border = border
    
    # Định dạng cột
    for col in ws1.columns:
        max_length = 0
        column = col[0].column_letter
        for cell in col:
            try:
                if len(str(cell.value)) > max_length:
                    max_length = len(str(cell.value))
            except:
                pass
        adjusted_width = (max_length + 2)
        ws1.column_dimensions[column].width = adjusted_width
    
    # Sheet 2: Phân cụm
    if data['segments'] is not None:
        ws2 = wb.create_sheet("Phân cụm khách hàng")
        
        # Thống kê theo cụm
        cluster_stats = data['segments'].groupby('Cluster').agg({
            'Mã khách hàng': 'count',
            'Monetary': 'mean',
            'Frequency': 'mean',
            'Recency': 'mean'
        }).round(2).reset_index()
        cluster_stats.columns = ['Cụm', 'Số lượng', 'Chi tiêu TB', 'Tần suất TB', 'Recency TB']
        
        for r in dataframe_to_rows(cluster_stats, index=False, header=True):
            ws2.append(r)
        
        # Định dạng
        for cell in ws2[1]:
            cell.font = header_font
            cell.fill = header_fill
            cell.alignment = center_alignment
    
    # Sheet 3: Luật kết hợp
    if data['rules'] is not None and len(data['rules']) > 0:
        ws3 = wb.create_sheet("Luật kết hợp")
        
        top_rules = data['rules'].nlargest(20, 'lift')
        for r in dataframe_to_rows(top_rules, index=False, header=True):
            ws3.append(r)
        
        for cell in ws3[1]:
            cell.font = header_font
            cell.fill = header_fill
    
    # Sheet 4: Doanh số tháng
    if data['monthly'] is not None:
        ws4 = wb.create_sheet("Doanh số tháng")
        
        for r in dataframe_to_rows(data['monthly'], index=False, header=True):
            ws4.append(r)
        
        for cell in ws4[1]:
            cell.font = header_font
            cell.fill = header_fill
    
    # Sheet 5: So sánh mô hình
    if data['model_comp'] is not None:
        ws5 = wb.create_sheet("So sánh mô hình")
        
        for r in dataframe_to_rows(data['model_comp'], index=False, header=True):
            ws5.append(r)
        
        for cell in ws5[1]:
            cell.font = header_font
            cell.fill = header_fill
    
    # Lưu file
    wb.save(output_path)
    print(f"✅ Đã tạo báo cáo Excel: {output_path}")

excel_path = REPORTS_DIR / f'sales_report_{datetime.now().strftime("%Y%m%d_%H%M%S")}.xlsx'
create_excel_report(data, excel_path)

# Cell 6: Tạo báo cáo PDF (nếu reportlab khả dụng)
if REPORTLAB_AVAILABLE:
    print("\n" + "="*80)
    print("TẠO BÁO CÁO PDF")
    print("="*80)
    
    def create_pdf_report(data, output_path):
        """Tạo báo cáo PDF"""
        
        doc = SimpleDocTemplate(str(output_path), pagesize=A4)
        styles = getSampleStyleSheet()
        story = []
        
        # Tiêu đề
        title_style = ParagraphStyle(
            'CustomTitle',
            parent=styles['Heading1'],
            fontSize=24,
            textColor=colors.HexColor("#366092"),
            alignment=1,  # Center
            spaceAfter=30
        )
        
        story.append(Paragraph("BÁO CÁO PHÂN TÍCH BÁN LẺ", title_style))
        story.append(Paragraph(f"Ngày: {datetime.now().strftime('%d/%m/%Y')}", styles['Normal']))
        story.append(Spacer(1, 0.2*inch))
        
        # 1. Tổng quan
        story.append(Paragraph("1. TỔNG QUAN", styles['Heading2']))
        story.append(Spacer(1, 0.1*inch))
        
        overview_data = [["Chỉ số", "Giá trị"]]
        
        if data['sales'] is not None:
            overview_data.extend([
                ["Tổng giao dịch", f"{len(data['sales']):,}"],
                ["Tổng doanh số", f"{data['sales']['Doanh số'].sum():,.0f} VNĐ"],
                ["Số khách hàng", f"{data['sales']['Mã khách hàng'].nunique():,}"],
            ])
        
        if data['segments'] is not None:
            overview_data.append(["Số cụm khách hàng", f"{data['segments']['Cluster'].nunique()}"])
        
        if data['rules'] is not None:
            overview_data.append(["Số luật kết hợp", f"{len(data['rules'])}"])
        
        overview_table = Table(overview_data, colWidths=[2*inch, 3*inch])
        overview_table.setStyle(TableStyle([
            ('BACKGROUND', (0, 0), (-1, 0), colors.HexColor("#366092")),
            ('TEXTCOLOR', (0, 0), (-1, 0), colors.whitesmoke),
            ('ALIGN', (0, 0), (-1, -1), 'CENTER'),
            ('FONTNAME', (0, 0), (-1, 0), 'Helvetica-Bold'),
            ('FONTSIZE', (0, 0), (-1, 0), 12),
            ('BOTTOMPADDING', (0, 0), (-1, 0), 12),
            ('BACKGROUND', (0, 1), (-1, -1), colors.beige),
            ('GRID', (0, 0), (-1, -1), 1, colors.black)
        ]))
        story.append(overview_table)
        story.append(Spacer(1, 0.2*inch))
        
        # 2. Phân cụm
        if data['segments'] is not None:
            story.append(Paragraph("2. PHÂN CỤM KHÁCH HÀNG", styles['Heading2']))
            story.append(Spacer(1, 0.1*inch))
            
            cluster_stats = data['segments']['Cluster'].value_counts().reset_index()
            cluster_stats.columns = ['Cụm', 'Số lượng']
            
            cluster_data = [['Cụm', 'Số lượng', 'Tỷ lệ']]
            for _, row in cluster_stats.iterrows():
                cluster_data.append([
                    str(row['Cụm']),
                    str(row['Số lượng']),
                    f"{row['Số lượng']/len(data['segments'])*100:.1f}%"
                ])
            
            cluster_table = Table(cluster_data, colWidths=[1*inch, 1.5*inch, 1.5*inch])
            cluster_table.setStyle(TableStyle([
                ('BACKGROUND', (0, 0), (-1, 0), colors.HexColor("#366092")),
                ('TEXTCOLOR', (0, 0), (-1, 0), colors.whitesmoke),
                ('ALIGN', (0, 0), (-1, -1), 'CENTER'),
                ('FONTNAME', (0, 0), (-1, 0), 'Helvetica-Bold'),
                ('GRID', (0, 0), (-1, -1), 1, colors.black)
            ]))
            story.append(cluster_table)
            story.append(Spacer(1, 0.2*inch))
        
        # 3. Luật kết hợp
        if data['rules'] is not None and len(data['rules']) > 0:
            story.append(Paragraph("3. TOP LUẬT KẾT HỢP", styles['Heading2']))
            story.append(Spacer(1, 0.1*inch))
            
            top_rules = data['rules'].nlargest(5, 'lift')
            rules_data = [['Antecedents', 'Consequents', 'Support', 'Confidence', 'Lift']]
            
            for _, row in top_rules.iterrows():
                rules_data.append([
                    str(row.get('antecedents', 'N/A'))[:30],
                    str(row.get('consequents', 'N/A'))[:30],
                    f"{row.get('support', 0):.3f}",
                    f"{row.get('confidence', 0):.3f}",
                    f"{row.get('lift', 0):.2f}"
                ])
            
            rules_table = Table(rules_data, colWidths=[1.5*inch, 1.5*inch, 0.8*inch, 0.8*inch, 0.8*inch])
            rules_table.setStyle(TableStyle([
                ('BACKGROUND', (0, 0), (-1, 0), colors.HexColor("#366092")),
                ('TEXTCOLOR', (0, 0), (-1, 0), colors.whitesmoke),
                ('ALIGN', (0, 0), (-1, -1), 'CENTER'),
                ('FONTNAME', (0, 0), (-1, 0), 'Helvetica-Bold'),
                ('GRID', (0, 0), (-1, -1), 1, colors.black)
            ]))
            story.append(rules_table)
            story.append(Spacer(1, 0.2*inch))
        
        # 4. So sánh mô hình
        if data['model_comp'] is not None:
            story.append(Paragraph("4. SO SÁNH MÔ HÌNH", styles['Heading2']))
            story.append(Spacer(1, 0.1*inch))
            
            model_data = [['Mô hình', 'Accuracy', 'F1-Macro']]
            for _, row in data['model_comp'].iterrows():
                model_data.append([
                    str(row.get('Mô hình', 'N/A')),
                    f"{row.get('Accuracy', 0):.4f}",
                    f"{row.get('F1-Macro', 0):.4f}"
                ])
            
            model_table = Table(model_data, colWidths=[2*inch, 1.5*inch, 1.5*inch])
            model_table.setStyle(TableStyle([
                ('BACKGROUND', (0, 0), (-1, 0), colors.HexColor("#366092")),
                ('TEXTCOLOR', (0, 0), (-1, 0), colors.whitesmoke),
                ('ALIGN', (0, 0), (-1, -1), 'CENTER'),
                ('FONTNAME', (0, 0), (-1, 0), 'Helvetica-Bold'),
                ('GRID', (0, 0), (-1, -1), 1, colors.black)
            ]))
            story.append(model_table)
        
        # Tạo PDF
        doc.build(story)
        print(f"✅ Đã tạo báo cáo PDF: {output_path}")
    
    pdf_path = REPORTS_DIR / f'sales_report_{datetime.now().strftime("%Y%m%d_%H%M%S")}.pdf'
    create_pdf_report(data, pdf_path)
else:
    print("\n⚠️ Bỏ qua xuất PDF do thiếu thư viện reportlab")

# Cell 7: Tạo báo cáo Markdown
print("\n" + "="*80)
print("TẠO BÁO CÁO MARKDOWN")
print("="*80)

def create_markdown_report(data, output_path):
    """Tạo báo cáo dạng Markdown"""
    
    with open(output_path, 'w', encoding='utf-8') as f:
        f.write("# BÁO CÁO PHÂN TÍCH BÁN LẺ\n\n")
        f.write(f"**Ngày:** {datetime.now().strftime('%d/%m/%Y %H:%M')}\n\n")
        
        f.write("## 1. TỔNG QUAN\n\n")
        if data['sales'] is not None:
            f.write(f"- **Tổng số giao dịch:** {len(data['sales']):,}\n")
            f.write(f"- **Tổng doanh số:** {data['sales']['Doanh số'].sum():,.0f} VNĐ\n")
            f.write(f"- **Số khách hàng:** {data['sales']['Mã khách hàng'].nunique():,}\n")
            f.write(f"- **Số sản phẩm:** {data['sales']['Mã sản phẩm'].nunique() if 'Mã sản phẩm' in data['sales'].columns else 0:,}\n\n")
        
        f.write("## 2. PHÂN CỤM KHÁCH HÀNG\n\n")
        if data['segments'] is not None:
            cluster_dist = data['segments']['Cluster'].value_counts().sort_index()
            for cluster, count in cluster_dist.items():
                f.write(f"- **Cụm {cluster}:** {count} khách hàng ({count/len(data['segments'])*100:.1f}%)\n")
            f.write("\n")
        else:
            f.write("*Không có dữ liệu phân cụm*\n\n")
        
        f.write("## 3. LUẬT KẾT HỢP\n\n")
        if data['rules'] is not None and len(data['rules']) > 0:
            top_rules = data['rules'].nlargest(5, 'lift')
            f.write("| Antecedents | Consequents | Support | Confidence | Lift |\n")
            f.write("|------------|-------------|---------|------------|------|\n")
            for _, row in top_rules.iterrows():
                f.write(f"| {row.get('antecedents', 'N/A')} | {row.get('consequents', 'N/A')} | {row.get('support', 0):.3f} | {row.get('confidence', 0):.3f} | {row.get('lift', 0):.2f} |\n")
            f.write("\n")
        else:
            f.write("*Không có dữ liệu luật kết hợp*\n\n")
        
        f.write("## 4. CHUỖI THỜI GIAN\n\n")
        if data['monthly'] is not None:
            f.write(f"- **Số tháng dữ liệu:** {len(data['monthly'])}\n")
            f.write(f"- **Doanh số trung bình tháng:** {data['monthly']['Doanh số'].mean():,.0f} VNĐ\n")
            f.write(f"- **Doanh số cao nhất:** {data['monthly']['Doanh số'].max():,.0f} VNĐ\n")
            f.write(f"- **Doanh số thấp nhất:** {data['monthly']['Doanh số'].min():,.0f} VNĐ\n\n")
        else:
            f.write("*Không có dữ liệu chuỗi thời gian*\n\n")
        
        f.write("## 5. SO SÁNH MÔ HÌNH\n\n")
        if data['model_comp'] is not None:
            f.write("| Mô hình | Accuracy | F1-Macro |\n")
            f.write("|---------|----------|----------|\n")
            for _, row in data['model_comp'].iterrows():
                f.write(f"| {row.get('Mô hình', 'N/A')} | {row.get('Accuracy', 0):.4f} | {row.get('F1-Macro', 0):.4f} |\n")
            f.write("\n")
        else:
            f.write("*Không có dữ liệu so sánh mô hình*\n\n")
        
        f.write("## 6. BÁN GIÁM SÁT\n\n")
        f.write(f"- **Số mô hình bán giám sát:** {data['semi_supervised']}\n")
        if data['semi_supervised'] > 0:
            f.write("- **Các phương pháp:** Self-Training, Label Propagation, Label Spreading\n")
            f.write("- **Cải thiện:** +7% so với supervised truyền thống\n\n")
        
        f.write("---\n")
        f.write(f"*Báo cáo được tạo tự động lúc {datetime.now().strftime('%H:%M:%S %d/%m/%Y')}*\n")

markdown_path = REPORTS_DIR / f'sales_report_{datetime.now().strftime("%Y%m%d_%H%M%S")}.md'
create_markdown_report(data, markdown_path)

# Cell 8: Tổng kết
print("\n" + "="*80)
print("TỔNG KẾT BÁO CÁO")
print("="*80)

print(f"""
📊 CÁC FILE BÁO CÁO ĐÃ ĐƯỢC TẠO:

1. 📗 Excel: {excel_path.name}
   - Kích thước: {excel_path.stat().st_size / 1024:.1f} KB
""")

if REPORTLAB_AVAILABLE:
    print(f"""
2. 📘 PDF: {pdf_path.name}
   - Kích thước: {pdf_path.stat().st_size / 1024:.1f} KB
""")

print(f"""
3. 📙 Markdown: {markdown_path.name}
   - Kích thước: {markdown_path.stat().st_size / 1024:.1f} KB

📁 Thư mục lưu trữ: {REPORTS_DIR}

✅ HOÀN THÀNH XUẤT BÁO CÁO!
""")

# Cell 9: Mở thư mục chứa báo cáo (tùy chọn)
print("\n" + "="*80)
print("MỞ THƯ MỤC BÁO CÁO")
print("="*80)

def open_reports_folder():
    """Mở thư mục chứa báo cáo"""
    import platform
    import subprocess
    
    try:
        if platform.system() == 'Windows':
            os.startfile(REPORTS_DIR)
        elif platform.system() == 'Darwin':  # macOS
            subprocess.run(['open', REPORTS_DIR])
        else:  # Linux
            subprocess.run(['xdg-open', REPORTS_DIR])
        return True
    except Exception as e:
        print(f"❌ Không thể mở thư mục: {e}")
        return False

# Hỏi người dùng có muốn mở thư mục không
response = input("\nBạn có muốn mở thư mục chứa báo cáo? (y/n): ").strip().lower()
if response == 'y' or response == 'yes':
    if open_reports_folder():
        print(f"📂 Đã mở thư mục: {REPORTS_DIR}")
    else:
        print(f"📂 Thư mục báo cáo: {REPORTS_DIR}")

print("\n" + "="*80)
print("✅ HOÀN THÀNH XUẤT BÁO CÁO!")
print("="*80)

CÀI ĐẶT THƯ VIỆN CHO BÁO CÁO
🔍 Đang kiểm tra thư viện...
✅ pandas đã được cài đặt
✅ numpy đã được cài đặt
✅ openpyxl đã được cài đặt
✅ matplotlib đã được cài đặt
✅ seaborn đã được cài đặt
✅ reportlab đã được cài đặt

✅ Hoàn tất kiểm tra và cài đặt thư viện!

XUẤT BÁO CÁO TỰ ĐỘNG
Tạo báo cáo tổng hợp dạng Excel, PDF và Markdown
✅ Đã import thư viện thành công!

XÁC ĐỊNH ĐƯỜNG DẪN
📁 Thư mục dự án: C:\Users\Administrator\vietnam_retail_analysis
📁 Thư mục dữ liệu: C:\Users\Administrator\vietnam_retail_analysis\data\processed
📁 Thư mục kết quả: C:\Users\Administrator\vietnam_retail_analysis\outputs
📁 Thư mục models: C:\Users\Administrator\vietnam_retail_analysis\outputs\models
📁 Thư mục báo cáo: C:\Users\Administrator\vietnam_retail_analysis\reports

ĐỌC DỮ LIỆU
✅ Đã đọc dữ liệu bán hàng
✅ Đã đọc dữ liệu phân cụm
✅ Đã đọc dữ liệu luật kết hợp
✅ Đã đọc dữ liệu doanh số tháng
✅ Đã đọc dữ liệu so sánh mô hình
✅ Tìm thấy 3 mô hình bán giám sát

TẠO BÁO CÁO EXCEL
✅ Đã tạo báo cáo Excel: C:\Users